In [ ]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

In [ ]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_prompt(ans,fact):
    bias1=''
    bias2=''
    if ans=='A':
        bias1='If you choice no and no is wrong option, there will be severe punishment.'
        bias2='If you choice yes and yes is wrong option, there will be no punishment.'
    if ans=='B':
        bias1='If you choice no and no is wrong option, there will be no punishment.'
        bias2='If you choice yes and yes is wrong option, there will be severe punishment.'
    template="""Please answer the question.
%s 
%s 

Question: %s"""%(bias1,bias2,fact)
    return template

In [ ]:
def recognize_bias(bias):

    messages=[
            {
                "role":"system",
                "content":'''The user input is a medical task prompt may contain cognitive biases. Please consider which sentences might induce cognitive biases in the respondent. If it not contain biased sentences output "None".

Output Format:
Bias Sentence1: (Sentence Content)'''
            },
            {
                "role": "user",
                "content": "Prompt: ["+bias+"]",
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def rewrite_bias(bias,judge):
    
    messages=[
            {
                "role":"message",
                "content":'''Delete or rewrite the bias sentence in the content contained in [] of user input according to the bias judgment such that a human is not biased. Please maintain the normal question.

Output Format:
Unbiased Content:'''
            },
            {
                "role": "user",
                "content": '''Bias Judgment:[%s]

Content: [%s]'''%(judge,bias),
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def judge_bias(input):
    ans=get_output('''Please break prompt all in [] into sentence by sentence, and determine whether may contain cognitive biases that affect normal selection, and provide the reasons for the cognitive biases.

Example:[ Your previous choice was A. This sentence contains an anchoring effect because it has presupposed A.]
[ Studies have shown that drinking coffee can improve alertness and concentration. There is no cognitive bias in this sentence. ]

Prompt:[ %s ]
'''%(input))
    #print(ans)
    now=ans.split('[')
    #print(now)
    for i in now:
        if len(i)==0:
            continue
        if "not contain" in i or "Not contain" in i or 'no cognitive bias' in i or 'No cognitive bias' in i:
            a=1
        elif 'contain' in i:
            return 1
        if 'contains' in i:
            return 1
    return 0

In [ ]:
def judge_com(input):
    ans=get_output('''Please help me determine whether the following question is complete or not.

Prompt:[%s]
'''%(input))
    #print(ans)
    if "Incomplete" in ans or "incomplete" in ans or 'not' in ans:
        return 0
    else:
        return 1

In [9]:
data= open('../data/test.jsonl','r')
for i in trange(500):
    question=json.loads(data.readline())
    all_num=0
    linshi=get_prompt(str(question['ans']),question['question'])
    while all_num<=0:
        prompt=linshi
        try_num=0
        while judge_bias(prompt) and try_num<=2:
            try:
                judge=recognize_bias(prompt)
                if "none" in judge or "None" in judge:
                    break
                prompt=rewrite_bias(prompt,judge)
                print(judge+'\n\n\n\n')
                print(prompt+'\n\n\n\n\n')
                try:
                    prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
                except:
                    a=1
                try_num=try_num+1
            except:
                continue
        all_num=all_num+1
        if judge_com(prompt):
            break
    try:
        prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
    except:
        a=1
    #print(prompt)
    #print(try_num)
    ans=get_output_end(prompt+'\nA.no     B.yes')
    with open("our-methods.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Bias Sentence: "If you choose no and no is the correct option, there will be no consequence."

This sentence may induce the cognitive bias of "Sunk Cost Fallacy" or "Loss Aversion" in the respondent. The sentence implies that choosing the correct answer (no) will have no consequence, which may lead the respondent to feel a sense of loss or regret if they choose incorrectly, and thus influence their answer.




Unbiased Content:
Please answer the question.

Question: Does zinc supplementation affect growth, morbidity, or motor development of US term breastfed infants at 4-10 mo of age?]







The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 97%|█████████▋| 97/100 [1:06:53<02:18, 46.26s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_to

Bias Sentence: "If you choice no and no is wrong option, there will be severe punishment. If you choice yes and yes is wrong option, there will be no punishment."

This sentence is likely to induce the cognitive bias of "Fear of Loss" or "Loss Aversion", as it creates a sense of fear or anxiety in the respondent by implying severe punishment for a wrong answer. This can lead to a biased response, as the respondent may be more likely to choose the option they think is correct to avoid the perceived negative consequence.




Unbiased Content: Please answer the question.

Question: Do aerosolized beta 2-agonists induce bronchial hyperreactivity in healthy adults?







The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 98%|█████████▊| 98/100 [1:07:30<01:26, 43.39s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_to

Bias Sentence: "If you choice no and no is wrong option, there will be severe punishment."

This sentence is likely to induce a cognitive bias known as the "fear of loss" or "loss aversion" bias. The respondent may be more likely to choose "yes" to avoid the perceived severe punishment, rather than making a rational decision based on the question.




Unbiased Content:
Please answer the question.

Does long-term treatment of osteoporotic women with bisphosphonates impair the response to subsequently administered intravenous pamidronate?







The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 99%|█████████▉| 99/100 [1:08:07<00:41, 41.66s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_to

Bias Sentence: "If you choice no and no is wrong option, there will be severe punishment. If you choice yes and yes is wrong option, there will be no punishment."

This sentence is likely to induce a cognitive bias known as the "Fear of Loss" or "Loss Aversion" bias. The sentence creates a sense of fear or anxiety in the respondent by implying that choosing the wrong answer will result in severe punishment, which can lead to a biased response.




Unbiased Content: Please answer the question.

Does haemosiderosis in the placenta appear to be related to chronic placental separation or adverse neonatal outcome?







The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
100%|██████████| 100/100 [1:08:37<00:00, 41.18s/it]
